In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.design_evaluation import scoring
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:329: UserWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.1.3. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [3]:
data

,CS textfield,BB textfield,Stack,Head angle,Head tube length textfield,Seat stay junction0,Seat tube length,Seat angle,DT Length,FORK0R,...,BELTorCHAIN OHCLASS: 1,RIM_STYLE front OHCLASS: DISC,RIM_STYLE front OHCLASS: SPOKED,RIM_STYLE front OHCLASS: TRISPOKE,RIM_STYLE rear OHCLASS: DISC,RIM_STYLE rear OHCLASS: SPOKED,RIM_STYLE rear OHCLASS: TRISPOKE,Seat tube type OHCLASS: 0,Seat tube type OHCLASS: 1,Seat tube type OHCLASS: 2
3553,381.00,55.002477,520.518278,72.000376,100.0,100.0,500.0,75.000376,619.064231,43.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1780,480.00,0.000000,530.900122,73.000000,152.7,45.0,489.8,90.000000,608.123819,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2322,425.00,62.000000,598.475325,72.200000,160.3,45.0,560.0,73.700000,657.859061,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1358,400.00,-37.500000,390.902939,67.500000,150.0,125.0,340.0,67.500000,469.378527,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
840,425.29,70.000000,571.512441,72.000000,124.0,45.0,565.8,74.300000,634.843371,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1174,430.00,65.000000,584.677620,72.000000,124.8,60.0,541.5,71.800000,632.625640,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4477,410.00,70.000000,658.278407,73.000000,226.4,55.0,610.0,73.000000,679.431932,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4659,420.00,72.000000,617.038218,72.500000,170.3,45.0,577.9,72.310741,659.860913,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1756,374.00,0.000000,482.702362,73.000000,115.0,55.0,301.0,74.000000,607.721727,45.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [4]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [5]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")

condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [6]:
eval_scores = evaluator(data_tens, condition)

tensor([0.4340])
tensor([[ 9.9117e+01, -3.6779e+01,  1.8928e+02],
        [ 9.4644e+01, -3.4435e+01, -7.8422e+00],
        [ 2.9260e+01, -3.4237e+01,  1.8924e+02],
        [ 1.2247e+02, -1.1688e+01,  1.8432e+02],
        [ 2.9892e+01, -4.9027e+01,  1.9509e+02],
        [ 7.7673e+01, -2.6844e+01,  1.9001e+02],
        [ 6.7051e+01, -1.6491e+01,  1.8152e+02],
        [ 9.0330e+01, -2.5651e+01,  1.8625e+02],
        [ 6.5364e+01, -3.5271e+01,  1.8864e+02],
        [ 6.4817e+01, -3.9970e+01,  1.9365e+02],
        [ 7.6776e+01, -1.4064e+01,  1.8248e+02],
        [ 3.0760e+01, -3.9965e+01,  1.9413e+02],
        [ 3.3381e+01, -3.2142e+01,  1.9337e+02],
        [ 8.1334e+01, -2.9909e+01,  1.8806e+02],
        [ 8.2249e+01, -3.3652e+01,  1.8918e+02],
        [ 8.2395e+01, -3.1898e+01,  1.9025e+02],
        [ 4.9416e+01, -3.6026e+01,  1.9364e+02],
        [ 1.5116e+02,  1.5633e+01,  1.8202e+02],
        [ 7.7613e+01, -2.5185e+01,  1.9167e+02],
        [ 3.4224e+01, -3.4652e+01,  1.8882e+02],
   

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [7]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [8]:
main_scorer = scoring.construct_scorer(MainScores, StandardEvaluations, data.columns)

In [9]:
main_scorer(data_tens, condition)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


tensor([0.4340])
tensor([[ 9.9117e+01, -3.6779e+01,  1.8928e+02],
        [ 9.4644e+01, -3.4435e+01, -7.8422e+00],
        [ 2.9260e+01, -3.4237e+01,  1.8924e+02],
        [ 1.2247e+02, -1.1688e+01,  1.8432e+02],
        [ 2.9892e+01, -4.9027e+01,  1.9509e+02],
        [ 7.7673e+01, -2.6844e+01,  1.9001e+02],
        [ 6.7051e+01, -1.6491e+01,  1.8152e+02],
        [ 9.0330e+01, -2.5651e+01,  1.8625e+02],
        [ 6.5364e+01, -3.5271e+01,  1.8864e+02],
        [ 6.4817e+01, -3.9970e+01,  1.9365e+02],
        [ 7.6776e+01, -1.4064e+01,  1.8248e+02],
        [ 3.0760e+01, -3.9965e+01,  1.9413e+02],
        [ 3.3381e+01, -3.2142e+01,  1.9337e+02],
        [ 8.1334e+01, -2.9909e+01,  1.8806e+02],
        [ 8.2249e+01, -3.3652e+01,  1.8918e+02],
        [ 8.2395e+01, -3.1898e+01,  1.9025e+02],
        [ 4.9416e+01, -3.6026e+01,  1.9364e+02],
        [ 1.5116e+02,  1.5633e+01,  1.8202e+02],
        [ 7.7613e+01, -2.5185e+01,  1.9167e+02],
        [ 3.4224e+01, -3.4652e+01,  1.8882e+02],
   

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ C:\Users\Lyle\AppData\Local\Temp\ipykernel_7828\3524032925.py:1 in <module>                      │
│                                                                                                  │
│ [Errno 2] No such file or directory:                                                             │
│ 'C:\\Users\\Lyle\\AppData\\Local\\Temp\\ipykernel_7828\\3524032925.py'                           │
│                                                                                                  │
│ c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\bik │
│ ed_commons\design_evaluation\scoring.py:174 in scorer                                            │
│                                                                                                  │
│   171 │   │   objective_scores = evaluation_scores[:, isobjective].detach().numpy()              │
│   172 │   │   ref_point_exp = np.expand_dims(ref_point, axis=0)                                  │
│   173 │   │   ref_point_exp = np.repeat(ref_point_exp, objective_scores.shape[0], axis=0)        │
│ ❱ 174 │   │   objective_scores[np.isnan(objective_scores)] = ref_point_exp[np.isnan(objective_   │
│   175 │   │   constraint_scores = evaluation_scores[:, ~isobjective].detach().numpy()            │
│   176 │   │   for scoring_function in scoring_functions:                                         │
│   177 │   │   │   raw = scoring_function.evaluate(designs, objective_scores, constraint_scores   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
IndexError: boolean index did not match indexed array along dimension 1; dimension is 10 but corresponding boolean 
dimension is 9